In [1]:
#!/usr/bin/env python
"""
LeafletFA Model Evaluation in Mouse 

This script:
1. Loads trained LeafletFA model outputs and associated data (mouse foundation smart-seq data)
2. Load the human foundation data and map junctions to human (via list of conserved junctions)
3. Apply the model to the human data
4. Save the predicted factor activities and factor usage
"""

import os
import sys
import glob
import pickle
import gzip
import warnings
from pathlib import Path
from collections import defaultdict
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from pyfaidx import Fasta

# Data analysis libraries
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import scipy.sparse as sp
from scipy.stats import spearmanr, pearsonr
from scipy.sparse import csr_matrix
from scipy.cluster.hierarchy import linkage, dendrogram
import scanpy as sc
import umap

# Machine learning libraries
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import (mean_squared_error, accuracy_score, r2_score, 
                           classification_report, confusion_matrix)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample

# Statistical modeling
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from mord import OrdinalRidge

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import ScalarFormatter
from adjustText import adjust_text

# Single-cell analysis libraries
import anndata as ad
import scanpy as sc

# Bioinformatics libraries
import gffutils
from tqdm import tqdm

# PyTorch setup
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("CUDA device name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.set_default_tensor_type("torch.FloatTensor" if device.type == "cpu" else "torch.cuda.FloatTensor")
torch.manual_seed(0)

# Configure plotting and warnings
sns.set_theme()
sc.set_figure_params(figsize=(7, 7), frameon=True, dpi=80, facecolor='white')
warnings.filterwarnings('ignore')

# =============================================================================
# Custom Module Imports
# =============================================================================

# Add custom module paths
leaflet_src_path = "/gpfs/commons/home/kisaev/Leaflet-private/src/"
utils_path = "/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/"

for path in [leaflet_src_path, utils_path]:
    if path not in sys.path:
        sys.path.append(path)

# Import LeafletFA modules
import BetaDirichletFactor.LeafletFA as LeafletFA
import BetaDirichletFactor.utils as utilsFA

# Import utility functions
from utils import load_model
from utils import *
from figure_plotting import *
from atse_viz import *

Torch version: 2.4.1.post300
CUDA available: True
CUDA device count: 2
CUDA device name: Tesla V100-PCIE-16GB
Using device: cuda


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1728241823685/work/torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)


Torch Version: 2.4.1.post300
CUDA Version: 12.0
Added /gpfs/commons/home/kisaev/LeafletFA-utils to sys.path
Visualization imports successful!


In [2]:
# Load genome
human_genome = Fasta("/gpfs/commons/datasets/controlled/BRAIN_NeMO/human-reference/gencode/GRCh38.primary_assembly.genome.fa")
mouse_genome = Fasta("/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/fasta/genome.fa")

# Load data first
MOUSE = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/ATSE_mapper/ATSE_files/MOUSE_FOUNDATION_ATSE_FILE_unanno_also_2025-10-01_21-36-40.txt.gz"
HUMAN = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/ATSE_mapper/ATSE_files/HUMAN_FOUNDATION_ATSE_FILE_unanno_also_2025-09-21_05-02-35.txt.gz"

print("Loading data...")
mouse_atse = pd.read_csv(MOUSE, sep="\t")
human_atse = pd.read_csv(HUMAN, sep="\t")

# Import utility functions - simple direct import
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/')
from utils import *
from figure_plotting import *

# Import all functions from /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/atse_viz.py
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/')
from atse_viz import *
import gffutils

gtf_file_mouse = "/gpfs/commons/groups/knowles_lab/Megan/encode_pacbio/2025_mouse_longread/2025_mouse_collapse_GRCm38/all_samples_sp_collapse_all_chr_full.gtf"
db_file_mouse = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/ATSE_mapper/genomes/lr_GRCm38.db"

# Load the database
db_mouse = gffutils.FeatureDB(db_file_mouse, keep_order=True)

gtf_file_human = "/gpfs/commons/groups/knowles_lab/Megan/encode_pacbio/paper_figures/isoform_gazers/all_samples_sp_collapse_all_chr_no_treatment_hashid_isoform_full.gtf"
db_file_human = "/gpfs/commons/groups/knowles_lab/Megan/encode_pacbio/paper_figures/isoform_gazers/long_read_hg38.db"

# Load the database
db_human = gffutils.FeatureDB(db_file_human, keep_order=True)

Loading data...


In [ ]:
# Load conserved junctions
conserved_junctions = pd.read_csv("/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Multi_Species_Splicing_Foundation/plots_2025-11-19/junction_mapping_mouse_human_with_annotations.csv")
conserved_junctions

,mouse_junction_id,human_junction_id,mouse_gene,human_gene,sequence_similarity,sequence_identity,coordinate_distance,conservation_status,motifs_conserved,mouse_donor_motif,human_donor_motif,mouse_acceptor_motif,human_acceptor_motif,mouse_annotation_status,human_annotation_status,conservation_confidence,human_gene_name
0,chr10_100080130_100080856_+,chr12_88515617_88516333_-,ENSMUSG00000019966,ENSG00000049130,0.942308,0.942308,0.0,conserved,True,GT,GT,AG,AG,both,both,high_confidence,KITLG
1,chr10_100080130_100087346_+,chr12_88507137_88516333_-,ENSMUSG00000019966,ENSG00000049130,0.846154,0.846154,0.0,conserved,True,GT,GT,AG,AG,both,both,high_confidence,KITLG
2,chr10_100080940_100087346_+,chr12_88507137_88515533_-,ENSMUSG00000019966,ENSG00000049130,0.903846,0.903846,0.0,conserved,True,GT,GT,AG,AG,both,both,high_confidence,KITLG
3,chr10_100512399_100513560_+,chr12_88115570_88117032_-,ENSMUSG00000019971,ENSG00000198707,0.836538,0.836538,0.0,conserved,True,GT,GT,AG,AG,five_prime,five_prime,high_confidence,CEP290
4,chr10_100512399_100513919_+,chr12_88115182_88117032_-,ENSMUSG00000019971,ENSG00000198707,0.875000,0.875000,0.0,conserved,True,GT,GT,AG,AG,both,both,high_confidence,CEP290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14997,chrY_1171002_1174727_-,chrY_13360528_13366266_-,ENSMUSG00000068457,ENSG00000183878,0.913462,0.913462,0.0,conserved,True,GT,GT,AG,AG,both,both,high_confidence,UTY
14998,chrY_1174854_1176490_-,chrY_13366393_13369255_-,ENSMUSG00000068457,ENSG00000183878,0.884615,0.884615,0.0,conserved,True,GT,GT,AG,AG,both,both,high_confidence,UTY
14999,chrY_1174854_1176495_-,chrY_13366393_13369260_-,ENSMUSG00000068457,ENSG00000183878,0.875000,0.875000,0.0,conserved,True,GT,GT,AG,AG,three_prime,three_prime,high_confidence,UTY
15000,chrY_927439_927602_+,chrY_19716111_19716278_-,ENSMUSG00000056673,ENSG00000012817,0.894231,0.894231,0.0,conserved,True,GT,GT,AG,AG,five_prime,five_prime,high_confidence,KDM5D


In [4]:
# Cell type tissue labels
cell_types = pd.read_csv("/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/NatureAging_PLOTTING_MOUSE/tissue_celltype_mapping.csv")

In [5]:
# Load full splice anndata for mouse and human
mouse_full_splice_adata = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/102025/model_ready_aligned_splicing_data_20251009_024406.h5ad"
full_splice_adata_mouse = ad.read_h5ad(mouse_full_splice_adata)
full_splice_adata_mouse.obs_names
MAX_JUNCTIONS = 5
full_splice_adata_mouse = full_splice_adata_mouse[:, full_splice_adata_mouse.var["num_junctions"] <= MAX_JUNCTIONS].copy()
full_splice_adata_mouse.var["junction_id_index"] = np.arange(full_splice_adata_mouse.shape[1])

In [6]:
# Load human full splice anndata
human_full_splice_adata = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/MODEL_INPUT/102025/model_ready_aligned_splicing_data_20251009_023419.h5ad"
full_splice_adata_human = ad.read_h5ad(human_full_splice_adata)
full_splice_adata_human.obs_names

Index(['F1S4_160106_001_B01', 'F1S4_160106_001_D01', 'F1S4_160106_001_E01',
       'F1S4_160106_001_G01', 'F1S4_160106_001_H01', 'F1S4_160106_002_C01',
       'F1S4_160106_002_D01', 'F1S4_160106_002_E01', 'F1S4_160106_002_F01',
       'F1S4_160106_002_G01',
       ...
       'TSP8_donor_B134140_P7_B134704_P7_Prostate_Epithelial',
       'TSP8_donor_B134140_P8_B134704_P8_Prostate_Epithelial',
       'TSP8_donor_B134140_P9_B134704_P9_Prostate_Epithelial',
       'TSP8_donor_B134141_B18_B134697_B18_Prostate_Immune',
       'TSP8_donor_B134141_B1_B134697_B1_Prostate_Immune',
       'TSP8_donor_B134141_C3_B134697_C3_Prostate_Immune',
       'TSP8_donor_B134141_E20_B134697_E20_Prostate_Immune',
       'TSP8_donor_B134141_G13_B134697_G13_Prostate_Immune',
       'TSP8_donor_B134141_H19_B134697_H19_Prostate_Immune',
       'TSP8_donor_B134141_H8_B134697_H8_Prostate_Immune'],
      dtype='object', name='cell_id_clean', length=76986)

In [7]:
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES ONLY
# =============================================================================

# Base directories
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION"
RESULTS_BASE_DIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/Mouse_Splicing_Foundation/model_train/MOUSE_FOUNDATION/results"
GE_ANNDATA_scVI_PATH = f"{BASE_DIR}/scVI/ge_adata_with_scvi_model_latent_20_20000_2025-10-09.h5ad"

# Reference files
AGING_GENES_PATH = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/27857814"
RBP_FILE_PATH = "/gpfs/commons/groups/knowles_lab/Karin/VanNostrand_2020_supptable1_41586_2020_2077_MOESM3_ESM.xlsx"

ATSE_FILE_PATH = (
    f"{BASE_DIR}/ATSE_mapper/ATSE_files/MOUSE_FOUNDATION_ATSE_FILE_unanno_also_2025-10-01_21-36-40.txt.gz"
)

# Load ATSE file
atse_df = pd.read_csv(ATSE_FILE_PATH, sep="\t")
atse_df["junction_annotation"] = "Novel_SS" 
atse_df.loc[atse_df["perfect_match_5_prime"].notna(), "junction_annotation"] = "5_prime_annotated"
atse_df.loc[atse_df["perfect_match_3_prime"].notna(), "junction_annotation"] = "3_prime_annotated"
atse_df.loc[atse_df["perfect_match_5_prime"].notna() & atse_df["perfect_match_3_prime"].notna(), "junction_annotation"] = "Both_SS_annotated"

# Load splicing data
ge_adata = ad.read_h5ad(GE_ANNDATA_scVI_PATH)
print(f"Done reading the ge_adata with scVI...")

# If ge_adata.var["gene_name"] is not in ge_adata.var_names, then add it
if "gene_name" not in ge_adata.var.columns:
    ge_adata.var["gene_name"] = ge_adata.var_names

# If ge_adata.obs doesn't have cell_id make it from cell_id_clean
if "cell_id" not in ge_adata.obs.columns:
    ge_adata.obs["cell_id"] = ge_adata.obs["cell_id_clean"]

# Load aging gene lists
aging_genes_mouse, aging_genes_human = load_aging_genes(AGING_GENES_PATH)
    
# Load RBP genes
rbps = load_rbp_genes(RBP_FILE_PATH)
rbps["mouse_gene_name"] = rbps["mouse_gene_name"].str.upper()
aging_genes_mouse = [g.upper() for g in aging_genes_mouse]

# if "mouse.id" is in splice_adata.obs rename it to donor_id 
print(f"Renaming mouse.id to donor_id in splice_adata.obs")
# Update gene annotations
ge_adata.var["RBP_gene"] = ge_adata.var["gene_name"].isin(rbps["mouse_gene_name"])
ge_adata.var["Aging_gene"] = ge_adata.var["gene_name"].isin(aging_genes_mouse)
rbps = rbps["mouse_gene_name"]

Done reading the ge_adata with scVI...
Loaded 330 aging-related genes (mouse)
Loaded 330 aging-related genes (human)
Loaded 356 RNA binding proteins
Renaming mouse.id to donor_id in splice_adata.obs


In [8]:
# Mapping file mouse --> human 
atse_mapping = conserved_junctions.copy()

# Subset atse_mapping to only include junctions that are in full_splice_adata_mouse.var["junction_id"]
atse_mapping = atse_mapping[atse_mapping["mouse_junction_id"].isin(full_splice_adata_mouse.var["junction_id"])]

# Subset atse_mapping to only include junctions that are in full_splice_adata_human.var["junction_id"]
atse_mapping = atse_mapping[atse_mapping["human_junction_id"].isin(full_splice_adata_human.var["junction_id"])]

In [9]:
# Now make subset_splice_adata_mouse and subset_splice_adata_human from full_splice_adata_mouse and full_splice_adata_human to only include junctions that are in atse_mapping["mouse_junction_id"] and atse_mapping["human_junction_id"] respectively
subset_splice_adata_mouse = full_splice_adata_mouse[:, full_splice_adata_mouse.var["junction_id"].isin(atse_mapping["mouse_junction_id"])].copy()
subset_splice_adata_human = full_splice_adata_human[:, full_splice_adata_human.var["junction_id"].isin(atse_mapping["human_junction_id"])].copy()

# Confirm number of junctions in each 
print(f"Number of junctions in subset_splice_adata_mouse: {subset_splice_adata_mouse.shape[1]}")
print(f"Number of junctions in subset_splice_adata_human: {subset_splice_adata_human.shape[1]}")

Number of junctions in subset_splice_adata_mouse: 13510
Number of junctions in subset_splice_adata_human: 13510


In [10]:
# For mouse we are going to keep the original junction_id_index since the model was trained on ALL junctions
# For human we need to reindex the junctions to match the mouse junctions
subset_splice_adata_human.var["junction_id_index"] = np.arange(subset_splice_adata_human.shape[1])

In [11]:
atse_mapping = atse_mapping[["human_junction_id", "mouse_junction_id", "mouse_gene", "human_gene"]].drop_duplicates()

In [12]:
atse_df_human = subset_splice_adata_human.var[["junction_id", "event_id"]]
atse_df_human.columns = ["human_junction_id", "human_event_id"]

atse_df_mouse = subset_splice_adata_mouse.var[["junction_id", "event_id"]]
atse_df_mouse.columns = ["mouse_junction_id", "mouse_event_id"]

# merge wtih atse_mapping
atse_mapping = atse_mapping.merge(atse_df_human, on="human_junction_id", how="left")
atse_mapping = atse_mapping.merge(atse_df_mouse, on="mouse_junction_id", how="left")

atse_mapping

,human_junction_id,mouse_junction_id,mouse_gene,human_gene,human_event_id,mouse_event_id
0,chr12_88515617_88516333_-,chr10_100080130_100080856_+,ENSMUSG00000019966,ENSG00000049130,ENSG00000049130.16_atse_3,ENSMUSG00000019966.18_atse_2
1,chr12_88507137_88516333_-,chr10_100080130_100087346_+,ENSMUSG00000019966,ENSG00000049130,ENSG00000049130.16_atse_3,ENSMUSG00000019966.18_atse_2
2,chr12_88507137_88515533_-,chr10_100080940_100087346_+,ENSMUSG00000019966,ENSG00000049130,ENSG00000049130.16_atse_3,ENSMUSG00000019966.18_atse_2
3,chr12_88115570_88117032_-,chr10_100512399_100513560_+,ENSMUSG00000019971,ENSG00000198707,ENSG00000198707.17_atse_4,ENSMUSG00000019971.10_atse_2
4,chr12_88115182_88117032_-,chr10_100512399_100513919_+,ENSMUSG00000019971,ENSG00000198707,ENSG00000198707.17_atse_4,ENSMUSG00000019971.10_atse_2
...,...,...,...,...,...,...
13505,chrY_13360528_13366266_-,chrY_1171002_1174727_-,ENSMUSG00000068457,ENSG00000183878,ENSG00000183878.16_atse_6,ENSMUSG00000068457.14_atse_6
13506,chrY_13366393_13369255_-,chrY_1174854_1176490_-,ENSMUSG00000068457,ENSG00000183878,ENSG00000183878.16_atse_5,ENSMUSG00000068457.14_atse_5
13507,chrY_13366393_13369260_-,chrY_1174854_1176495_-,ENSMUSG00000068457,ENSG00000183878,ENSG00000183878.16_atse_5,ENSMUSG00000068457.14_atse_5
13508,chrY_19716111_19716278_-,chrY_927439_927602_+,ENSMUSG00000056673,ENSG00000012817,ENSG00000012817.16_atse_4,ENSMUSG00000056673.14_atse_6


In [13]:
# Count actual junctions per event_id
actual_junction_counts = subset_splice_adata_mouse.var.groupby("event_id")["junction_id"].nunique()

# Get expected junction counts per event_id
expected_junction_counts = subset_splice_adata_mouse.var.groupby("event_id")["num_junctions"].first()

# Create comparison dataframe
junction_comparison = pd.DataFrame({
    'event_id': actual_junction_counts.index,
    'actual_junctions': actual_junction_counts.values,
    'expected_junctions': expected_junction_counts.values
})

# Check for mismatches
junction_comparison['match'] = (junction_comparison['actual_junctions'] == 
                               junction_comparison['expected_junctions'])

# Keep just those where match is True 
junction_comparison = junction_comparison[junction_comparison['match']]
print(f"Number of remaining ATSEs to look at: {len(junction_comparison)}")

Number of remaining ATSEs to look at: 2658


In [14]:
# Main parameters to change
MODEL_TRAIN_DATE = "2025-11-13"
MODEL_ANALYSIS_DATE = "2025-11-19"
PARAM_ID = 0

# =============================================================================
# AUTO-GENERATED PATHS - DON'T EDIT BELOW THIS LINE
# =============================================================================

# Core result directories
PARAM_RESULTS_DIR = f"{RESULTS_BASE_DIR}/{MODEL_TRAIN_DATE}/{MODEL_ANALYSIS_DATE}/param_id_{PARAM_ID}"
DATA_DIR = f"{PARAM_RESULTS_DIR}/data"

# Model outputs
MODEL_OUTPUTS_DIR = f"{BASE_DIR}/Leaflet/leafletFAmodel/{MODEL_TRAIN_DATE}"
MODEL_PATH = f"{MODEL_OUTPUTS_DIR}/run_{PARAM_ID}/leafletfa_model.pkl.gz"

# Main data files from downstream analysis of model 
SPLICE_ADATA_PATH = f"{DATA_DIR}/splice_adata_PHI_psi_var_obs.h5ad"
PI_VALUES_PATH = f"{DATA_DIR}/PI_values.npy"
DIFF_SPL_PATH = f"{DATA_DIR}/differential_splicing_results.csv"

# Output directory for current analysis
OUTPUT_DIR = f"{PARAM_RESULTS_DIR}/analysis_outputs"

pi = np.load(PI_VALUES_PATH)
leaflet_model = load_model(MODEL_PATH)
diff_spl = pd.read_csv(DIFF_SPL_PATH)

Loading model to device: cuda


In [15]:
# -------------------------------------------------------------------------------------
# ESTIMATE FACTOR ACTIVITIES FROM PSI VALUES
# -------------------------------------------------------------------------------------
psi = leaflet_model["psi_learned"]

# Find the indices of the mouse junctions in the splice_adata.var["junction_id"]
psi_subset = psi[:, subset_splice_adata_mouse.var["junction_id_index"].values]
psi_subset.shape

(20, 13510)

In [16]:
subset_splice_adata_mouse.var["mouse_junction_id"] = subset_splice_adata_mouse.var["junction_id"]

In [17]:
# Need to make sure the human junctions (the ones that are conserved) are in the same order as the mouse junctions
# reorder atse_mapping to match the order of the mouse junctions
# use mouse_junction_id column in atse_mapping to reorder
# use splice_adata_mouse.var["mouse_junction_id"] to reorder
atse_mapping = atse_mapping[atse_mapping["mouse_junction_id"].isin(subset_splice_adata_mouse.var["mouse_junction_id"])]

# assert they are in the same order
assert np.all(atse_mapping["mouse_junction_id"].values == subset_splice_adata_mouse.var["mouse_junction_id"].values)

In [18]:
# Now order subset_human to be in the same order as the mouse junctions
subset_splice_adata_human.var["human_junction_id"] = subset_splice_adata_human.var["junction_id"]

# Filter the AnnData object's columns (junctions) to include only those present in the mapping table
conserved_human_ids = atse_mapping["human_junction_id"].values
subset_splice_adata_human = subset_splice_adata_human[:, subset_splice_adata_human.var["human_junction_id"].isin(conserved_human_ids)].copy()

# 2. **Crucial Step: Reorder the columns** (junctions)
# Use the desired order from the mapping table to reindex the AnnData object.
desired_order = atse_mapping["human_junction_id"].values

# Get the indices in subset_splice_adata_human.var that correspond to the desired order
# This ensures that both the .X matrix and .var are correctly rearranged.
order_map = pd.Series(
    np.arange(subset_splice_adata_human.shape[1]), 
    index=subset_splice_adata_human.var["human_junction_id"]
)
reorder_indices = order_map.loc[desired_order].values

# Apply the reordering
subset_splice_adata_human = subset_splice_adata_human[:, reorder_indices].copy()

# 3. Final Assertion (The check for correctness)
# The order of human_junction_id in the AnnData object MUST now match the order
# of human_junction_id in the *already-ordered* atse_mapping table.
assert np.all(subset_splice_adata_human.var["human_junction_id"].values == atse_mapping["human_junction_id"].values)

# Re-add index for subsequent operations (if needed)
subset_splice_adata_human.var["junction_id_index"] = np.arange(subset_splice_adata_human.shape[1])

In [19]:
subset_splice_adata_mouse.var[["junction_id", "junction_id_index"]]

,junction_id,junction_id_index
2,chr10_100080130_100080856_+,2
3,chr10_100080130_100087346_+,3
4,chr10_100080940_100087346_+,4
14,chr10_100512399_100513560_+,14
15,chr10_100512399_100513919_+,15
...,...,...
89780,chrY_1171002_1174727_-,70058
89782,chrY_1174854_1176490_-,70060
89783,chrY_1174854_1176495_-,70061
89822,chrY_927439_927602_+,70093


In [20]:
import os
import scipy.sparse
# Assuming 'atse_mapping', 'subset_splice_adata_mouse', and 'subset_splice_adata_human' are already defined and loaded.

# --- Directory Setup ---
output_dir = "/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation"
mouse_human_transfer_learning_dir = f"{output_dir}/mouse_human_transfer_learning"

# Create directory if it doesn't exist
if not os.path.exists(mouse_human_transfer_learning_dir):
    os.makedirs(mouse_human_transfer_learning_dir)
print(f"Directory check complete: {mouse_human_transfer_learning_dir}")

# --- Define File Paths ---
atse_mapping_path = f"{mouse_human_transfer_learning_dir}/atse_mapping.csv"
subset_splice_adata_mouse_path = f"{mouse_human_transfer_learning_dir}/subset_splice_adata_mouse.h5ad"
subset_splice_adata_human_path = f"{mouse_human_transfer_learning_dir}/subset_splice_adata_human.h5ad"

# --- Save atse_mapping (CSV) ---
print(f"Saving mapping file to: {atse_mapping_path}")
atse_mapping.to_csv(atse_mapping_path, index=False)

# --- Save Mouse AnnData ---
print(f"Saving mouse AnnData to: {subset_splice_adata_mouse_path}")
subset_splice_adata_mouse.write_h5ad(subset_splice_adata_mouse_path)

# --- Fix and Save Human AnnData ---

# IORegistryError Fix: Convert any problematic layer from coo_matrix to csr_matrix
# Checking for both Cluster_Counts (previous issue) and Junction_Counts (current issue)
problematic_layers = ['Cluster_Counts', 'Junction_Counts']
for layer_key in problematic_layers:
    if layer_key in subset_splice_adata_human.layers:
        layer_data = subset_splice_adata_human.layers[layer_key]
        if isinstance(layer_data, scipy.sparse.coo_matrix):
            print(f"Fixing IORegistryError: Converting layer '{layer_key}' from COO to CSR format.")
            subset_splice_adata_human.layers[layer_key] = layer_data.tocsr()
        else:
            print(f"Layer '{layer_key}' is present and in a compatible format.")
    else:
        print(f"Note: Layer '{layer_key}' was not found in human AnnData layers.")

# Save the Human AnnData
print(f"Saving human AnnData to: {subset_splice_adata_human_path}")
subset_splice_adata_human.write_h5ad(subset_splice_adata_human_path)

print("\nAll files saved successfully.")

Directory check complete: /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/mouse_human_transfer_learning
Saving mapping file to: /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/mouse_human_transfer_learning/atse_mapping.csv
Saving mouse AnnData to: /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/mouse_human_transfer_learning/subset_splice_adata_mouse.h5ad
Note: Layer 'Cluster_Counts' was not found in human AnnData layers.
Note: Layer 'Junction_Counts' was not found in human AnnData layers.
Saving human AnnData to: /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/mouse_human_transfer_learning/subset_splice_adata_human.h5ad

All files saved successfully.
